# ALS validation tuning

Compare a 16-configuration ALS grid using a validation split made only from the deterministic outer training set. For each fit, measure validation RMSE plus Precision@10 and NDCG@10 against validation movies rated at least 4.0. The held-out test partition is never used by this experiment.

The grid holds `maxIter=10`, ALS seed `42`, and `coldStartStrategy="drop"` fixed while varying rank and regularization. The proposal's baseline (`rank=50`, `regParam=0.1`) remains unchanged in `als-baseline.ipynb`; this notebook only reports how it compares. Spark runs with `local[2]` and the project's approximately 1 GB driver heap. The compact MovieLens 1M factors and per-fit cleanup keep the search bounded; if a configuration fails due to memory, record the failed configuration and available heap rather than increasing concurrency or silently omitting it.


## Start Spark and verify the runtime

Use Spark 3.5.9, Scala 2.12.18, Java 17, and two local worker threads. The synthetic count verifies that Spark can execute before reading MovieLens data.


In [1]:
import $ivy.`org.apache.spark::spark-sql:3.5.9`
import $ivy.`org.apache.spark::spark-mllib:3.5.9`

import org.apache.spark.sql.SparkSession


import $ivy.$
import $ivy.$
import org.apache.spark.sql.SparkSession

In [2]:
val spark = SparkSession.builder()
  .appName("CS 7265 - ALS Validation Tuning")
  .master("local[2]")
  .config("spark.driver.host", "127.0.0.1")
  .config("spark.driver.bindAddress", "127.0.0.1")
  .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
val scalaVersion = scala.util.Properties.versionNumberString
val javaVersion = System.getProperty("java.version")
val environmentCheck = spark.range(3).count()
assert(spark.version == "3.5.9")
assert(scalaVersion == "2.12.18")
assert(System.getProperty("java.specification.version") == "17")
assert(environmentCheck == 3L)
println(s"Spark: ${spark.version}; Scala: $scalaVersion; Java: $javaVersion")
println(s"PASS: environment check on $environmentCheck synthetic rows")


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/11/15 20:28:10 WARN Utils: Your hostname, Jrdns-MacBook-Air.local resolves to a loopback address: 127.0.0.1; using 192.168.0.194 instead (on interface en0)
26/11/15 20:28:10 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
26/11/15 20:28:10 INFO SparkContext: Running Spark version 3.5.9
26/11/15 20:28:10 INFO SparkContext: OS info Mac OS X, 26.5.1, aarch64
26/11/15 20:28:10 INFO SparkContext: Java version 17.0.20.1
26/11/15 20:28:11 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/11/15 20:28:11 INFO ResourceUtils: ==============================================================
26/11/15 20:28:11 INFO ResourceUtils: No custom resources configured for spark.driver.
26/11/15 20:28:11 INFO ResourceUtils: ==============================================================
26/11/15 20:28:11 INFO SparkContext: Submitt

Spark: 3.5.9; Scala: 2.12.18; Java: 17.0.20.1
PASS: environment check on 3 synthetic rows


spark: SparkSession = org.apache.spark.sql.SparkSession@70b52663
scalaVersion: String = "2.12.18"
javaVersion: String = "17.0.20.1"
environmentCheck: Long = 3L

## Load ratings and construct only the outer training partition

Use the canonical per-user split from `popularity-profiles.ipynb`: order by `xxhash64(42, userId, movieId)` then `movieId`, and assign `floor(0.8 * userRatingCount)` rows to training. This cell creates only `trainingRatings`; it does not construct, count, cache, or otherwise consume test rows. The validation split below is derived solely from `trainingRatings`.


In [3]:
case class Rating(userId: Int, movieId: Int, rating: Float, timestamp: Long)

class RatingParser extends (String => Rating) with Serializable {
  def apply(line: String): Rating = {
    val fields = line.split("::")
    require(fields.length == 4, "Expected four fields per MovieLens rating")
    Rating(fields(0).toInt, fields(1).toInt, fields(2).toFloat, fields(3).toLong)
  }
}


defined class Rating
defined class RatingParser

In [4]:
val ratingsPath = Seq(
  "data/ml-1m/ratings.dat",
  "../data/ml-1m/ratings.dat"
).find(path => new java.io.File(path).isFile).getOrElse {
  throw new IllegalArgumentException("Place MovieLens 1M ratings.dat in data/ml-1m/ and run from the repository root or notebooks folder.")
}
val repositoryRoot = new java.io.File(ratingsPath).getCanonicalFile
  .getParentFile.getParentFile.getParentFile
val ratingsRDD = spark.sparkContext.textFile(ratingsPath).map(new RatingParser)
import spark.implicits._
val ratingsDF = ratingsRDD.toDF().cache()
val ratingCount = ratingsDF.count()
require(ratingCount == 1000209L, s"Expected 1000209 MovieLens ratings, found $ratingCount")
println(s"PASS: loaded $ratingCount ratings")


PASS: loaded 1000209 ratings


ratingsPath: String = "../data/ml-1m/ratings.dat"
repositoryRoot: java.io.File = /Users/jrdn/Documents/GitHub/history-aware-popularity-calibration
ratingsRDD: org.apache.spark.rdd.RDD[Rating] = MapPartitionsRDD[9] at map at cmd4.sc:9
import spark.implicits._
ratingsDF: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [userId: int, movieId: int ... 2 more fields]
ratingCount: Long = 1000209L

In [5]:
import org.apache.spark.sql.expressions.Window
import org.apache.spark.sql.functions._

val trainingFraction = 0.8
val splitSeed = 42L
val perUser = Window.partitionBy("userId")
val ratingsWithCounts = ratingsDF.withColumn("userRatingCount", count("*").over(perUser))
val ratingsWithKeys = ratingsWithCounts.withColumn(
  "shuffleKey", xxhash64(lit(splitSeed), col("userId"), col("movieId")))
val shuffledPerUser = Window.partitionBy("userId")
  .orderBy(col("shuffleKey"), col("movieId"))
val rankedRatings = ratingsWithKeys.withColumn("ratingPosition", row_number().over(shuffledPerUser))
val ratingsWithCutoff = rankedRatings.withColumn(
  "trainingCount", floor(col("userRatingCount") * trainingFraction))
val trainingRatings = ratingsWithCutoff
  .filter(col("ratingPosition") <= col("trainingCount"))
  .select("userId", "movieId", "rating", "timestamp")
  .persist(org.apache.spark.storage.StorageLevel.MEMORY_AND_DISK)
val trainingRatingCount = trainingRatings.count()
assert(trainingRatingCount == 797758L,
  s"Expected 797758 outer-training ratings, found $trainingRatingCount")
println(s"PASS: outer training partition contains $trainingRatingCount ratings; test rows are not evaluated")


PASS: outer training partition contains 797758 ratings; test rows are not evaluated


import org.apache.spark.sql.expressions.Window
import org.apache.spark.sql.functions._
trainingFraction: Double = 0.8
splitSeed: Long = 42L
perUser: org.apache.spark.sql.expressions.WindowSpec = org.apache.spark.sql.expressions.WindowSpec@1a5ad5c2
ratingsWithCounts: org.apache.spark.sql.package.DataFrame = [userId: int, movieId: int ... 3 more fields]
ratingsWithKeys: org.apache.spark.sql.package.DataFrame = [userId: int, movieId: int ... 4 more fields]
shuffledPerUser: org.apache.spark.sql.expressions.WindowSpec = org.apache.spark.sql.expressions.WindowSpec@1b3996be
rankedRatings: org.apache.spark.sql.package.DataFrame = [userId: int, movieId: int ... 5 more fields]
ratingsWithCutoff: org.apache.spark.sql.package.DataFrame = [userId: int, movieId: int ... 6 more fields]
trainingRatings: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [userId: int, movieId: int ... 2 more fields]
trainingRatingCount: Long = 797758L

## Create the validation split from trainingRatings only

Within the outer training partition, independently order each user's rows by `xxhash64(7, userId, movieId)` then `movieId`; use the first `floor(0.8 * n)` rows to fit and the rest for validation. Confirm the partitions are nonempty and disjoint, and that their counts add back to the outer training count.


In [6]:
import org.apache.spark.sql.DataFrame

// Per-user hash split; the input is explicitly the outer training partition.
def perUserSplit(df: DataFrame, seed: Long, fraction: Double): (DataFrame, DataFrame) = {
  require(fraction > 0.0 && fraction < 1.0, "Split fraction must be strictly between 0 and 1.")
  val ranked = df
    .withColumn("n", count("*").over(Window.partitionBy("userId")))
    .withColumn("k", xxhash64(lit(seed), col("userId"), col("movieId")))
    .withColumn("pos", row_number().over(
      Window.partitionBy("userId").orderBy(col("k"), col("movieId"))))
    .withColumn("cut", floor(col("n") * fraction))
  val fit = ranked.filter(col("pos") <= col("cut"))
    .select("userId", "movieId", "rating", "timestamp")
  val validation = ranked.filter(col("pos") > col("cut"))
    .select("userId", "movieId", "rating", "timestamp")
  (fit, validation)
}

val (trainFit, validation) = perUserSplit(trainingRatings, seed = 7L, fraction = 0.8)
trainFit.persist(org.apache.spark.storage.StorageLevel.MEMORY_AND_DISK)
validation.persist(org.apache.spark.storage.StorageLevel.MEMORY_AND_DISK)
val trainFitCount = trainFit.count()
val validationCount = validation.count()
val validationOverlap = trainFit.select("userId", "movieId")
  .join(validation.select("userId", "movieId"), Seq("userId", "movieId"), "inner")
  .limit(1).count()
assert(trainFitCount > 0L && validationCount > 0L,
  s"Validation partitions must be nonempty: fit=$trainFitCount, validation=$validationCount")
assert(trainFitCount + validationCount == trainingRatingCount,
  s"Validation split lost rows: fit=$trainFitCount + validation=$validationCount, outer=$trainingRatingCount")
assert(validationOverlap == 0L, "Per-user fit and validation partitions overlap.")
println(s"PASS: seed=7 training-only split trainFit=$trainFitCount, validation=$validationCount, overlap=$validationOverlap")
ratingsDF.unpersist(blocking = true)
trainingRatings.unpersist(blocking = true)


PASS: seed=7 training-only split trainFit=635723, validation=162035, overlap=0


import org.apache.spark.sql.DataFrame
defined function perUserSplit
trainFit: DataFrame = [userId: int, movieId: int ... 2 more fields]
validation: DataFrame = [userId: int, movieId: int ... 2 more fields]
res6_3: DataFrame = [userId: int, movieId: int ... 2 more fields]
res6_4: DataFrame = [userId: int, movieId: int ... 2 more fields]
trainFitCount: Long = 635723L
validationCount: Long = 162035L
validationOverlap: Long = 0L
res6_12: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [userId: int, movieId: int ... 2 more fields]
res6_13: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [userId: int, movieId: int ... 2 more fields]

## Run the grid with bounded per-fit memory

The only model input is `trainFit`. RMSE is evaluated over validation ratings that ALS can score (`coldStartStrategy="drop"`). For ranking metrics, score each validation user/movie pair, rank predictions by score (movie ID breaks ties), and compare each eligible user's top 10 with that user's validation movies rated `>= 4.0`. Eligible users with no scored validation predictions remain in the ranking evaluation with an empty recommendation list, so cold-start drops are not hidden.

Each model is fit sequentially. Materialized predictions and model factor tables are unpersisted in `finally`, references are released, and garbage collection is requested before the next fit. `fitSeconds` reports model-fitting time; `evaluationSeconds` reports prediction and metric time. Results are saved in a new timestamped run folder using create-new semantics.


In [7]:
import org.apache.spark.ml.evaluation.RegressionEvaluator
import org.apache.spark.ml.recommendation.ALS
import org.apache.spark.mllib.evaluation.RankingMetrics
import org.apache.spark.sql.functions._

case class TuningResult(
  rank: Int,
  regParam: Double,
  maxIter: Int,
  seed: Long,
  rmse: Double,
  precisionAt10: Double,
  ndcgAt10: Double,
  scoredValidationRatings: Long,
  eligibleUsers: Long,
  fitSeconds: Double,
  evaluationSeconds: Double
)

val ranks = Seq(10, 20, 50, 100)
val regParams = Seq(0.01, 0.05, 0.1, 0.2)
val maxIter = 10
val alsSeed = 42L
val listSize = 10
val validationRelevant = validation.filter(col("rating") >= 4.0)
  .select("userId", "movieId").cache()
val relevantCount = validationRelevant.count()
val eligibleUsers = validationRelevant.select("userId").distinct().cache()
val eligibleUserCount = eligibleUsers.count()
require(relevantCount > 0L && eligibleUserCount > 0L,
  s"Expected relevant validation items; relevant=$relevantCount, users=$eligibleUserCount")
println(s"PASS: $relevantCount validation items rated >= 4 from $eligibleUserCount eligible users")

val rmseEvaluator = new RegressionEvaluator()
  .setMetricName("rmse").setLabelCol("rating").setPredictionCol("prediction")
val resultRows = scala.collection.mutable.ArrayBuffer.empty[TuningResult]

def evaluateConfiguration(rank: Int, regParam: Double): TuningResult = {
  val als = new ALS()
    .setUserCol("userId").setItemCol("movieId").setRatingCol("rating")
    .setImplicitPrefs(false)
    .setRank(rank).setMaxIter(maxIter).setRegParam(regParam).setSeed(alsSeed)
    .setColdStartStrategy("drop")

  val fitStarted = System.nanoTime()
  val model = als.fit(trainFit)
  val fitSeconds = (System.nanoTime() - fitStarted).toDouble / 1e9
  var scoredValidation: DataFrame = null
  try {
    val evaluationStarted = System.nanoTime()
    scoredValidation = model.transform(validation)
      .filter(col("prediction").isNotNull && !isnan(col("prediction")))
      .persist(org.apache.spark.storage.StorageLevel.MEMORY_AND_DISK)
    val scoredValidationRatings = scoredValidation.count()
    require(scoredValidationRatings > 0L,
      s"No validation ratings scored for rank=$rank, regParam=$regParam")
    val rmse = rmseEvaluator.evaluate(scoredValidation)

    val predictedLists = scoredValidation.groupBy("userId")
      .agg(slice(sort_array(collect_list(struct(
        (-col("prediction")).as("negativePrediction"), col("movieId").as("movieId")))),
        1, listSize).as("ranked"))
      .select(col("userId"), expr("transform(ranked, x -> x.movieId)").as("predicted"))
    val actualLists = validationRelevant.groupBy("userId")
      .agg(collect_set("movieId").as("actual"))
    val rankingRows = actualLists.join(eligibleUsers, Seq("userId"), "inner")
      .join(predictedLists, Seq("userId"), "left")
      .select(coalesce(col("predicted"), array().cast("array<int>")).as("predicted"),
        col("actual"))
    val rankingRdd = rankingRows.rdd.map { row =>
      (row.getSeq[Int](0).toArray, row.getSeq[Int](1).toArray)
    }
    val rankingMetrics = new RankingMetrics(rankingRdd)
    val precisionAt10 = rankingMetrics.precisionAt(listSize)
    val ndcgAt10 = rankingMetrics.ndcgAt(listSize)
    val evaluationSeconds = (System.nanoTime() - evaluationStarted).toDouble / 1e9
    require(Seq(rmse, precisionAt10, ndcgAt10).forall(x => !x.isNaN && !x.isInfinity),
      s"Non-finite metric for rank=$rank, regParam=$regParam")
    TuningResult(rank, regParam, maxIter, alsSeed, rmse,
      precisionAt10, ndcgAt10, scoredValidationRatings, eligibleUserCount,
      fitSeconds, evaluationSeconds)
  } finally {
    if (scoredValidation != null) scoredValidation.unpersist(blocking = true)
    model.userFactors.unpersist(blocking = true)
    model.itemFactors.unpersist(blocking = true)
    System.gc()
  }
}

for (rank <- ranks; regParam <- regParams) {
  val result = evaluateConfiguration(rank, regParam)
  resultRows += result
  println(f"PASS: rank=$rank%3d regParam=$regParam%.2f fit=${result.fitSeconds}%.2fs " +
    f"eval=${result.evaluationSeconds}%.2fs RMSE=${result.rmse}%.6f " +
    f"P@10=${result.precisionAt10}%.6f NDCG@10=${result.ndcgAt10}%.6f " +
    f"scored=${result.scoredValidationRatings}%d")
}
assert(resultRows.size == ranks.size * regParams.size,
  s"Expected ${ranks.size * regParams.size} grid results, found ${resultRows.size}")
println(s"PASS: completed ${resultRows.size} distinct ALS configurations")


PASS: 93510 validation items rated >= 4 from 6007 eligible users
PASS: rank= 10 regParam=0.01 fit=15.31s eval=6.89s RMSE=0.918000 P@10=0.654054 NDCG@10=0.863899 scored=161998
PASS: rank= 10 regParam=0.05 fit=7.80s eval=4.52s RMSE=0.872653 P@10=0.662128 NDCG@10=0.881434 scored=161998
PASS: rank= 10 regParam=0.10 fit=6.35s eval=3.62s RMSE=0.874961 P@10=0.663076 NDCG@10=0.885368 scored=161998
PASS: rank= 10 regParam=0.20 fit=6.54s eval=3.83s RMSE=0.919466 P@10=0.650375 NDCG@10=0.868871 scored=161998
PASS: rank= 20 regParam=0.01 fit=7.64s eval=3.69s RMSE=0.984904 P@10=0.644232 NDCG@10=0.847630 scored=161998
PASS: rank= 20 regParam=0.05 fit=7.76s eval=3.83s RMSE=0.879870 P@10=0.661395 NDCG@10=0.879105 scored=161998
PASS: rank= 20 regParam=0.10 fit=8.45s eval=4.57s RMSE=0.872624 P@10=0.663792 NDCG@10=0.887057 scored=161998
PASS: rank= 20 regParam=0.20 fit=12.33s eval=4.91s RMSE=0.918674 P@10=0.650591 NDCG@10=0.868953 scored=161998
PASS: rank= 50 regParam=0.01 fit=18.14s eval=9.74s RMSE=1.063

import org.apache.spark.ml.evaluation.RegressionEvaluator
import org.apache.spark.ml.recommendation.ALS
import org.apache.spark.mllib.evaluation.RankingMetrics
import org.apache.spark.sql.functions._
defined class TuningResult
ranks: Seq[Int] = List(10, 20, 50, 100)
regParams: Seq[Double] = List(0.01, 0.05, 0.1, 0.2)
maxIter: Int = 10
alsSeed: Long = 42L
listSize: Int = 10
validationRelevant: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [userId: int, movieId: int]
relevantCount: Long = 93510L
eligibleUsers: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [userId: int]
eligibleUserCount: Long = 6007L
rmseEvaluator: RegressionEvaluator = RegressionEvaluator: uid=regEval_be258d1abf1c, metricName=rmse, throughOrigin=false
resultRows: collection.mutable.ArrayBuffer[TuningResult] = ArrayBuffer(
  TuningResult(
    10,
    0.01,
    10,
    42L,
    0.9179998656942832,
    0.6540536041285168,
    0.8638988562485636,
    161998L,
    6007L,
    15.313795083,
    6.88622762

## Compare settings and save results

Display every grid row sorted by validation RMSE and by validation NDCG@10. “Optimal across all objectives” means the proposal's `(rank=50, regParam=0.1)` ties for the best RMSE, Precision@10, and NDCG@10 simultaneously; because these objectives may disagree, the notebook also states its standing for each measure separately. This report does not change the ALS baseline configuration.


In [8]:
val results = resultRows.toSeq.toDF().cache()
assert(results.count() == 16L, "Expected exactly 16 grid rows.")
assert(results.select("rank", "regParam").distinct().count() == 16L,
  "Grid contains duplicate or missing rank/regParam configurations.")

println("Validation results sorted by RMSE (ascending):")
results.orderBy(col("rmse").asc, col("rank").asc, col("regParam").asc)
  .show(20, truncate = false)
println("Validation results sorted by NDCG@10 (descending):")
results.orderBy(col("ndcgAt10").desc, col("rank").asc, col("regParam").asc)
  .show(20, truncate = false)

val resultByConfig = resultRows.map(r => (r.rank, r.regParam) -> r).toMap
val proposal = resultByConfig((50, 0.1))
val bestRmse = resultRows.map(_.rmse).min
val bestPrecision = resultRows.map(_.precisionAt10).max
val bestNdcg = resultRows.map(_.ndcgAt10).max
val metricTieTolerance = 1e-12
val proposalBestRmse = math.abs(proposal.rmse - bestRmse) <= metricTieTolerance
val proposalBestPrecision = math.abs(proposal.precisionAt10 - bestPrecision) <= metricTieTolerance
val proposalBestNdcg = math.abs(proposal.ndcgAt10 - bestNdcg) <= metricTieTolerance
val optimalOnAll = proposalBestRmse && proposalBestPrecision && proposalBestNdcg
println(s"Proposal rank=50/regParam=0.1: RMSE=${proposal.rmse}, P@10=${proposal.precisionAt10}, NDCG@10=${proposal.ndcgAt10}")
println(s"Best grid values: RMSE=$bestRmse; P@10=$bestPrecision; NDCG@10=$bestNdcg")
println(s"Proposal standing: RMSE ${if (proposalBestRmse) "best (tied)" else "not best"}; " +
  s"P@10 ${if (proposalBestPrecision) "best (tied)" else "not best"}; " +
  s"NDCG@10 ${if (proposalBestNdcg) "best (tied)" else "not best"}.")
println(if (optimalOnAll)
  "FINDING: rank 50 / regParam 0.1 is optimal across all three measured validation objectives."
else
  "FINDING: rank 50 / regParam 0.1 is not uniformly optimal across RMSE, Precision@10, and NDCG@10; see the sorted tables for the trade-offs.")

val tuningRunId = java.time.Instant.now().toString.replace(":", "-")
val tuningOutputDir = new java.io.File(repositoryRoot, s"outputs/als-tuning/$tuningRunId")
require(tuningOutputDir.mkdirs(),
  s"Could not create unique tuning output folder (it may already exist): ${tuningOutputDir.getPath}")
val gridPath = new java.io.File(tuningOutputDir, "grid.csv").toPath
val writer = java.nio.file.Files.newBufferedWriter(gridPath,
  java.nio.charset.StandardCharsets.UTF_8,
  java.nio.file.StandardOpenOption.CREATE_NEW,
  java.nio.file.StandardOpenOption.WRITE)
try {
  writer.write("rank,regParam,maxIter,seed,rmse,precisionAt10,ndcgAt10,scoredValidationRatings,eligibleUsers,fitSeconds,evaluationSeconds")
  writer.newLine()
  resultRows.foreach { r =>
    writer.write(Seq(r.rank.toString, r.regParam.toString, r.maxIter.toString,
      r.seed.toString, r.rmse.toString, r.precisionAt10.toString, r.ndcgAt10.toString,
      r.scoredValidationRatings.toString, r.eligibleUsers.toString,
      r.fitSeconds.toString, r.evaluationSeconds.toString).mkString(","))
    writer.newLine()
  }
} finally {
  writer.close()
}
assert(java.nio.file.Files.size(gridPath) > 0L, "Grid CSV was not written.")
println(s"PASS: saved ${resultRows.size} grid results to ${gridPath.toString}")


Validation results sorted by RMSE (ascending):
+----+--------+-------+----+------------------+------------------+------------------+-----------------------+-------------+------------+-----------------+
|rank|regParam|maxIter|seed|rmse              |precisionAt10     |ndcgAt10          |scoredValidationRatings|eligibleUsers|fitSeconds  |evaluationSeconds|
+----+--------+-------+----+------------------+------------------+------------------+-----------------------+-------------+------------+-----------------+
|20  |0.1     |10     |42  |0.8726242012981735|0.6637922423838856|0.8870570243359525|161998                 |6007         |8.450559584 |4.573271292      |
|10  |0.05    |10     |42  |0.8726534159209436|0.6621275178957884|0.8814344104103309|161998                 |6007         |7.795448208 |4.523424083      |
|50  |0.1     |10     |42  |0.8741652953481678|0.665173963709006 |0.8892055697667226|161998                 |6007         |15.09893875 |6.501639         |
|100 |0.1     |10     |

results: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [rank: int, regParam: double ... 9 more fields]
resultByConfig: Map[(Int, Double), TuningResult] = Map(
  (100, 0.01) -> TuningResult(
    100,
    0.01,
    10,
    42L,
    1.0504887952368382,
    0.6289162643582488,
    0.8417167168807568,
    161998L,
    6007L,
    34.467943,
    8.489147125
  ),
  (20, 0.2) -> TuningResult(
    20,
    0.2,
    10,
    42L,
    0.918673814551589,
    0.650590977193275,
    0.8689526156040068,
    161998L,
    6007L,
    12.32725525,
    4.909067333
  ),
  (100, 0.2) -> TuningResult(
    100,
    0.2,
    10,
    42L,
    0.920555906430091,
    0.6506742134176794,
    0.8691704248437905,
    161998L,
    6007L,
    35.17408975,
    8.708168666
...
proposal: TuningResult = TuningResult(
  50,
  0.1,
  10,
  42L,
  0.8741652953481678,
  0.665173963709006,
  0.8892055697667226,
  161998L,
  6007L,
  15.09893875,
  6.501639
)
bestRmse: Double = 0.8726242012981735
bestPrecision: Double =

## Stop Spark

Unpersist the shared validation data and stop the local session after saving the results.


In [9]:
validationRelevant.unpersist(blocking = true)
eligibleUsers.unpersist(blocking = true)
trainFit.unpersist(blocking = true)
validation.unpersist(blocking = true)
results.unpersist(blocking = true)
spark.stop()
println("PASS: cached validation data released and Spark session stopped")


PASS: cached validation data released and Spark session stopped


res9_0: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [userId: int, movieId: int]
res9_1: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [userId: int]
res9_2: DataFrame = [userId: int, movieId: int ... 2 more fields]
res9_3: DataFrame = [userId: int, movieId: int ... 2 more fields]
res9_4: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [rank: int, regParam: double ... 9 more fields]